# Задание 2 (основная часть). Распознавание лиц: CE loss и ArcFace

Это **третья стадия** пайплайна — сеть-распознаватель. Она принимает выровненное лицо (выход
Задания 1) и выдаёт **эмбеддинг** — вектор, у которого важны не абсолютные значения, а
направление: лица одного человека должны давать близкие по косинусу векторы, разных — далёкие.

**Что делаем:**

1. Берём выровненный датасет из Задания 1 (`aligned/train`, `aligned/val`).
2. Бэкбон — **ResNet‑50, предобученный на ImageNet** (распознаванию лиц он не учился — это запрещено правилами).
3. Обучаем базовую модель на **Cross‑Entropy** (каждая личность = класс), цель — `accuracy ≥ 0.7`.
4. Реализуем **ArcFace** (Additive Angular Margin) и обучаем такую же модель на нём.
5. Достаём эмбеддинги (голову-классификатор выкидываем) и **сравниваем CE vs ArcFace**.

> Бэкбон выбран `ResNet‑50` (стандарт для face recognition). Код параметризован — при нехватке
> GPU замените на `resnet18`/`resnet34` одной строкой в `build_backbone`.

## 0. Конфигурация и данные

Используем те же пути и тот же выровненный датасет, что собрали в Задании 1.

In [ ]:
import os, math, time, random, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

WORK_DIR    = "/content/drive/MyDrive/face_project"     # как в Задании 1
ALIGNED_DIR = os.path.join(WORK_DIR, "aligned")
EMB_DIM     = 512                                       # размер эмбеддинга
IMG_SIZE    = 112
BATCH       = 128
# from google.colab import drive; drive.mount('/content/drive')

**Аугментации и нормализация.** Лица уже выровнены, поэтому сильная геометрия не нужна — берём
лёгкий флип и color jitter. Нормализуем по статистикам ImageNet (бэкбон предобучен на них).

Классификация считается на **тех же личностях**, но на **разных фото**: `aligned/train` для
обучения, `aligned/val` для валидационной accuracy (closed-set). Личности `query`/`distractors`
здесь не используются — они пойдут в метрику ID‑Rate (доп. задание 1).

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_set = ImageFolder(os.path.join(ALIGNED_DIR, "train"), transform=train_tf)
val_set   = ImageFolder(os.path.join(ALIGNED_DIR, "val"),   transform=eval_tf)
# val должен использовать те же class->idx, что train (одни и те же личности)
val_set.class_to_idx = train_set.class_to_idx
val_set.samples = [(p, train_set.class_to_idx[os.path.basename(os.path.dirname(p))])
                   for p, _ in val_set.samples]

NUM_CLASSES = len(train_set.classes)
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH, shuffle=False, num_workers=2)
print("классов (личностей):", NUM_CLASSES, "| train:", len(train_set), "| val:", len(val_set))

## 1. Бэкбон → эмбеддинг

Берём `ResNet‑50` с весами ImageNet, убираем «голову» классификации на 1000 классов ImageNet и
ставим свою: `Linear(2048 → 512) → BatchNorm`. **BatchNorm в конце** (без последующего ReLU) —
частый приём в face recognition: он стабилизирует распределение эмбеддингов перед нормализацией.

Эмбеддинг можно вернуть «сырым» (для CE‑классификатора) или **L2‑нормализованным** (для косинусных
сравнений и для ArcFace).

In [ ]:
def build_backbone(name="resnet50", pretrained=True):
    fn = getattr(torchvision.models, name)
    weights = "IMAGENET1K_V2" if pretrained else None
    net = fn(weights=weights)
    in_feats = net.fc.in_features
    net.fc = nn.Identity()
    return net, in_feats

class FaceEmbeddingNet(nn.Module):
    '''Бэкбон (ImageNet) -> эмбеддинг EMB_DIM. ВАЖНО: не предобучен на лицах.'''
    def __init__(self, emb_dim=EMB_DIM, backbone="resnet50", pretrained=True):
        super().__init__()
        self.backbone, in_feats = build_backbone(backbone, pretrained)
        self.embedding = nn.Sequential(nn.Linear(in_feats, emb_dim), nn.BatchNorm1d(emb_dim))
    def forward(self, x, normalize=False):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb) if normalize else emb

**Sanity-check бэкбона** (реальный вывод ячейки): формы и то, что L2-нормализация даёт единичные
векторы. `ResNet‑50` после замены головы — ≈ 24.6M параметров.

In [1]:
_m = FaceEmbeddingNet(pretrained=False).eval()
_x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    _e, _en = _m(_x), _m(_x, normalize=True)
print("input:", tuple(_x.shape), "-> embedding:", tuple(_e.shape))
print("normalized norms:", [round(float(v),3) for v in _en.norm(dim=1)])
print("params: %.1fM" % (sum(p.numel() for p in _m.parameters())/1e6))
del _m, _x

input: (4, 3, 112, 112) -> embedding: (4, 512)
normalized norms: [1.0, 1.0, 1.0, 1.0]
params: 24.6M


## 2. Базовая модель на Cross‑Entropy

Простейший подход: эмбеддинг → линейный слой на `NUM_CLASSES` → стандартная кросс-энтропия.
После обучения эмбеддинги уже неплохо разделяют людей (в т.ч. незнакомых), но CE **никак не
контролирует геометрию**: она лишь добивается разделимости классов, не сближая эмбеддинги
одного человека и не раздвигая разных. Это мы и улучшим ArcFace’ом.

In [ ]:
class CEModel(nn.Module):
    def __init__(self, backbone="resnet50", pretrained=True):
        super().__init__()
        self.embnet = FaceEmbeddingNet(backbone=backbone, pretrained=pretrained)
        self.classifier = nn.Linear(EMB_DIM, NUM_CLASSES)
    def forward(self, x):
        emb = self.embnet(x)
        return emb, self.classifier(emb)

@torch.no_grad()
def accuracy(model_fwd, loader):
    '''model_fwd(x) -> logits. Возвращает top-1 accuracy.'''
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model_fwd(x)
        correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total

In [ ]:
def train_classifier(model, head_forward, params, epochs=15, lr=1e-3, tag="model"):
    '''Общий цикл обучения под CE. head_forward(x,y)->logits даёт гибкость для ArcFace.'''
    model.to(DEVICE)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_acc, hist = 0.0, {"loss": [], "val_acc": []}
    for ep in range(epochs):
        model.train(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = head_forward(x, y)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * x.size(0)
        sched.step()
        model.eval()
        va = accuracy(lambda x: head_forward(x, None), val_loader)
        hist["loss"].append(running/len(train_set)); hist["val_acc"].append(va)
        if va > best_acc:
            best_acc = va; torch.save(model.state_dict(), os.path.join(WORK_DIR, f"{tag}_best.pt"))
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  loss {hist['loss'][-1]:.4f}  val_acc {va:.4f}")
    print(f"[{tag}] лучшая val_acc = {best_acc:.4f}")
    return hist

In [ ]:
ce_model = CEModel(backbone="resnet50", pretrained=True)

def ce_forward(x, y=None):
    _, logits = ce_model(x)
    return logits

ce_hist = train_classifier(ce_model, ce_forward, ce_model.parameters(),
                           epochs=15, lr=1e-3, tag="ce")

## 3. ArcFace (Additive Angular Margin)

CE расставляет классы как угодно, лишь бы они разделялись. ArcFace меняет правила: эмбеддинги и
веса классов **нормируются на сферу**, а к углу до «своего» класса добавляется **угловой отступ
`m`**. Сети приходится оставлять запас по углу — классы становятся компактнее и дальше друг от друга.

Пошагово (это просто модифицированный софтмакс, а лосс — та же CE):

1. нормируем эмбеддинг `x` и веса классов `W`: на выходе `cosθ_j = x̂ · Ŵ_j`;
2. для **целевого** класса заменяем `cosθ_y → cos(θ_y + m)`;
3. умножаем все логиты на масштаб `s` и подаём в обычную `CrossEntropy`.

$$L_{ArcFace}=\frac{-1}{N}\sum_i \log\frac{e^{s\cos(\theta_{y_i}+m)}}{e^{s\cos(\theta_{y_i}+m)}+\sum_{j\ne y_i}e^{s\cos\theta_j}}$$

Типичные гиперпараметры: `s = 30`, `m = 0.5`. Реализуем через тождество
`cos(θ+m)=cosθ·cos m − sinθ·sin m` (численно стабильнее, чем `arccos`).
Оригинал: [Deng et al., 2019](https://arxiv.org/pdf/1801.07698.pdf).

In [ ]:
class ArcFace(nn.Module):
    def __init__(self, emb_dim, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.empty(num_classes, emb_dim)); nn.init.xavier_normal_(self.W)
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)          # порог: при θ+m>π используем линейную ветку
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.W)).clamp(-1, 1)   # cosθ_j, [B,C]
        if labels is None:                       # режим оценки: чистый косинус-классификатор
            return cosine * self.s
        sine = torch.sqrt((1.0 - cosine ** 2).clamp(min=1e-9))
        phi = cosine * self.cos_m - sine * self.sin_m                # cos(θ + m)
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)   # монотонность при больших θ
        one_hot = F.one_hot(labels, cosine.size(1)).float()
        logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return logits

**Sanity-check ArcFace** (реальный вывод, `torch.manual_seed(0)`): логиты имеют форму `[B, C]`,
градиент доходит до эмбеддингов, и логит «своего» класса **не больше** обычного косинуса —
значит угловой отступ действительно штрафует целевой класс.

In [2]:
torch.manual_seed(0)
_emb = torch.randn(8, EMB_DIM, requires_grad=True); _lab = torch.randint(0, 100, (8,))
_arc = ArcFace(EMB_DIM, 100)
_logits = _arc(_emb, _lab); _loss = F.cross_entropy(_logits, _lab); _loss.backward()
print("ArcFace logits:", tuple(_logits.shape), "| CE loss: %.4f" % _loss.item(),
      "| grad finite:", bool(torch.isfinite(_emb.grad).all()))
with torch.no_grad():
    _cos = F.linear(F.normalize(_emb), F.normalize(_arc.W)).clamp(-1, 1)
    _ok = (_logits[torch.arange(8), _lab] / _arc.s <= _cos[torch.arange(8), _lab] + 1e-5).all()
print("margin check (target logit <= plain cosine):", bool(_ok))
del _emb, _arc, _logits

ArcFace logits: (8, 100) | CE loss: 19.7074 | grad finite: True
margin check (target logit <= plain cosine): True


### Обучение с ArcFace

Бэкбон тот же (`FaceEmbeddingNet`), но вместо линейного классификатора — слой `ArcFace`.
При обучении подаём метки (нужен угловой отступ), при валидации меток нет — `ArcFace` работает как
обычный косинусный классификатор, и мы честно считаем accuracy.

In [ ]:
arc_embnet = FaceEmbeddingNet(backbone="resnet50", pretrained=True).to(DEVICE)
arc_head   = ArcFace(EMB_DIM, NUM_CLASSES, s=30.0, m=0.5).to(DEVICE)

class ArcWrapper(nn.Module):
    def __init__(self, embnet, head): super().__init__(); self.embnet, self.head = embnet, head
arc_model = ArcWrapper(arc_embnet, arc_head)

def arc_forward(x, y=None):
    emb = arc_embnet(x)
    return arc_head(emb, y)            # y=None на валидации -> s*cosine

arc_hist = train_classifier(arc_model, arc_forward,
                            list(arc_embnet.parameters()) + list(arc_head.parameters()),
                            epochs=15, lr=1e-3, tag="arcface")

## 4. Сравнение CE vs ArcFace

Сравниваем по двум осям: (1) валидационная accuracy по эпохам; (2) **качество эмбеддингов** —
насколько хорошо они разделяют людей. Для (2) считаем на валидации среднее косинусное сходство
**внутри** класса (должно быть высоким) и **между** классами (должно быть низким); разрыв между
ними — грубая мера «качества» пространства эмбеддингов. Как правило, у ArcFace разрыv заметно больше.

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(ce_hist["val_acc"],  marker="o", label="CE")
plt.plot(arc_hist["val_acc"], marker="s", label="ArcFace")
plt.axhline(0.7, ls="--", c="gray", label="порог 0.7")
plt.xlabel("эпоха"); plt.ylabel("val accuracy"); plt.title("CE vs ArcFace: accuracy")
plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
@torch.no_grad()
def embed_dataset(embnet, loader):
    embnet.eval(); E, Y = [], []
    for x, y in loader:
        E.append(F.normalize(embnet(x.to(DEVICE))).cpu()); Y.append(y)
    return torch.cat(E), torch.cat(Y)

def intra_inter(E, Y, max_per_class=10):
    '''Среднее косинусное сходство внутри классов и между классами (по подвыборке пар).'''
    import itertools, random as rnd
    by = {}
    for i, c in enumerate(Y.tolist()): by.setdefault(c, []).append(i)
    intra, inter = [], []
    for c, idx in by.items():
        idx = idx[:max_per_class]
        for a, b in itertools.combinations(idx, 2):
            intra.append(float(E[a] @ E[b]))
    cls = list(by.keys())
    for _ in range(2000):
        c1, c2 = rnd.sample(cls, 2)
        inter.append(float(E[by[c1][0]] @ E[by[c2][0]]))
    return np.mean(intra), np.mean(inter)

for tag, embnet in [("CE", ce_model.embnet), ("ArcFace", arc_embnet)]:
    E, Y = embed_dataset(embnet, val_loader)
    a, b = intra_inter(E, Y)
    print(f"{tag:8s}  intra-class cos = {a:.3f}  inter-class cos = {b:.3f}  разрыв = {a-b:.3f}")

## 5. Сохранение и итоги

Для распознавания нам нужны **только эмбеддинги** — голову (`Linear`/`ArcFace`) выкидываем.
Сохраняем веса бэкбонов обеих моделей: они пригодятся в Задании 3 (пайплайн) и в доп. заданиях.

In [ ]:
torch.save(ce_model.embnet.state_dict(),  os.path.join(WORK_DIR, "embnet_ce.pt"))
torch.save(arc_embnet.state_dict(),        os.path.join(WORK_DIR, "embnet_arcface.pt"))
print("Сохранены эмбеддинг-сети: embnet_ce.pt, embnet_arcface.pt")

**Выводы (шаблон — впишите свои числа):**

* Обе модели должны достигать `val accuracy ≥ 0.7` (closed-set по личностям обучения).
* У ArcFace разрыв `intra − inter` по косинусу обычно заметно больше → эмбеддинги «кучнее» внутри
  человека и дальше между людьми. Именно это даёт прирост на **незнакомых** лицах, что мы измерим
  численно в доп. задании 1 (Identification Rate).
* Здесь же удобно описать ваш реальный опыт: какой `lr`/`m`/`s` сработал, не «разваливалось» ли
  обучение ArcFace в начале (частая проблема — большой `m` сразу; помогает прогрев/меньший `m`).

➡️ Дальше: **Задание 3 (`3_Pipeline.ipynb`)** — собираем детектор + выравнивание + распознавание в один пайплайн.